# Parameterised pentagon spiral winding

A planar spiral winding whose turns follow a rounded pentagon rather than a circle.

The base shape is an **inner pentagon**, whose five vertices act as the centre poles for
the cornering arcs. The wire then walks outwards turn by turn: an arc around a pole, a
straight run along the offset edge to the next pole, and so on. Every corner advances the
radius by `pitch / 5`, so a full lap of five corners advances exactly one `pitch`.

This alternates the `a` and `l` directives:

- `a x_z y_z 1 1` rounds a corner about the pole at `(x_z, y_z)` in the positive direction,
  linearly interpolating the radius from that pole as it sweeps, which is what gives the
  turn-to-turn advance.
- `l 1` runs straight along the edge, offset outwards by the current radius.

The `a` directive is what makes this possible: `s` can only spiral about the origin, so a
five-pole winding would otherwise need the whole pattern translated between corners.


## 1. Define the winding

In [ ]:
import numpy as np

# ---- parameters ----
r_pentagon = 0.030    # circumradius of the inner pentagon (m) -- the five corner poles
r_cs       = 0.00045  # wire cross-section radius (m)
pitch      = 0.0015   # radial advance per turn (m)
n_turns    = 6        # number of turns
r_corner_0 = pitch    # corner arc radius of the innermost turn (m)

n_sides = 5

if pitch <= 2 * r_cs:
    print("warning: pitch %g m is not greater than the wire diameter %g m, turns will overlap"
          % (pitch, 2 * r_cs))

# ---- the inner pentagon: five corner poles, counter-clockwise ----
theta = np.arange(n_sides) * 2.0 * np.pi / n_sides
poles = np.stack([r_pentagon * np.cos(theta), r_pentagon * np.sin(theta)], axis=1)

def outward_normal(a, b):
    """Outward unit normal of the edge a -> b of a counter-clockwise polygon."""
    u = (b - a) / np.linalg.norm(b - a)
    return np.array([u[1], -u[0]])

normals = np.array([outward_normal(poles[i], poles[(i + 1) % n_sides])
                    for i in range(n_sides)])

# ---- walk the spiral: arc round a pole, straight along the offset edge, repeat ----
# The point arriving at pole i sits at radius r along normals[i - 1]; the arc about pole i
# sweeps it round to radius r + dr along normals[i], so each corner advances one fifth of
# a pitch and the five corners of a lap advance one whole pitch.
dr = pitch / n_sides
r = r_corner_0
commands = [("l", None, poles[0] + r * normals[-1])]
for turn in range(n_turns):
    for i in range(n_sides):
        r += dr
        commands.append(("a", poles[i], poles[i] + r * normals[i]))
        commands.append(("l", None, poles[(i + 1) % n_sides] + r * normals[i]))

# ---- write the wire definition file ----
with open("wire_def", "w") as file_out_def:
    file_out_def.write("1\r\n")
    file_out_def.write("%g %d\r\n" % (r_cs, len(commands)))
    for mode, pole, point in commands:
        if mode == "a":
            file_out_def.write("a %.9g %.9g 1 1\r\n" % (pole[0], pole[1]))
        else:
            file_out_def.write("l 1\r\n")
        file_out_def.write("%.9g %.9g 0\r\n" % (point[0], point[1]))

print("wrote wire_def: %d turns, %d commands, outermost corner radius %.4f m"
      % (n_turns, len(commands), r))
print()
print("".join(open("wire_def").readlines()[:7]).replace("\r", ""), "...", sep="")

### Plot the winding

The solver writes the segmented trajectory to `pentagon_0.trj`, so the winding is plotted
after the solver has run in section 2. Here we just preview the key points that were
written to `wire_def`, with the inner pentagon dashed.

In [ ]:
import matplotlib.pyplot as plt

key_points = np.array([point for _, _, point in commands])

figure, ax = plt.subplots(figsize=(6, 6))
ax.plot(key_points[:, 0], key_points[:, 1], "o-", ms=2, lw=0.8, label="key points")
ax.plot(np.append(poles[:, 0], poles[0, 0]), np.append(poles[:, 1], poles[0, 1]),
        "r--", lw=0.8, label="inner pentagon (corner poles)")
ax.set_aspect("equal")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title("Key points of the pentagon winding")
ax.legend(loc="upper right", fontsize=8)
plt.show()

## 2. Solve for self-inductance and total length

The solver is run in its default `-l` mode, which writes `pentagon.lmat`. For a single
wire that file holds a 1x1 L matrix -- the self-inductance in henries -- followed by a
last line giving the total wire length in metres.

In [ ]:
import os

status = os.system("../../../build/ivsolver wire_def pentagon")
if status != 0:
    raise RuntimeError("ivsolver failed with status %d" % status)

# The .lmat file is n rows of the L matrix followed by one row of wire lengths, so for a
# single wire it is one row holding L and one holding the length. Parse it by row rather
# than with loadtxt, which would flatten the 1x1 case into a single vector.
rows = [line.split() for line in open("pentagon.lmat") if line.strip()]
n_wires = len(rows) - 1
l_matrix = np.array([[float(value) for value in row] for row in rows[:n_wires]])
lengths = np.array([float(value) for value in rows[n_wires]])

self_inductance = l_matrix[0, 0]
total_length = lengths[0]

print("self-inductance: %.4f uH" % (self_inductance * 1e6))
print("total length:    %.4f m" % total_length)

### Cross-check the length

Each straight run is exactly one side of the pentagon, and each corner sweeps the exterior
angle `2*pi/5` at a radius that grows linearly across the arc, so the total length has a
closed form to compare the solver against. The residual is the discretisation error of
approximating the arcs by chords of one wire diameter.

In [ ]:
side = np.linalg.norm(poles[1] - poles[0])
n_corners = n_turns * n_sides
r_mean = r_corner_0 + (np.arange(n_corners) + 0.5) * dr
length_analytic = n_corners * side + (2 * np.pi / n_sides) * r_mean.sum()

print("analytic length: %.6f m" % length_analytic)
print("solver length:   %.6f m" % total_length)
print("difference:      %.4f %%" % (100 * (total_length - length_analytic) / length_analytic))

### Plot the segmented trajectory

`pentagon_0.trj` is the trajectory the solver actually integrated over, segmented into
steps of one wire diameter.

In [ ]:
trajectory = np.loadtxt("pentagon_0.trj")

figure, ax = plt.subplots(figsize=(6, 6))
ax.plot(trajectory[:, 0], trajectory[:, 1], lw=0.9)
ax.plot(np.append(poles[:, 0], poles[0, 0]), np.append(poles[:, 1], poles[0, 1]),
        "r--", lw=0.8, label="inner pentagon (corner poles)")
ax.set_aspect("equal")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title("%d-turn pentagon winding, %.2f uH, %.3f m"
             % (n_turns, self_inductance * 1e6, total_length))
ax.legend(loc="upper right", fontsize=8)
figure.savefig("winding.png", dpi=110, bbox_inches="tight")
plt.show()

print("%d trajectory points" % len(trajectory))